# Two architectures on the same split: ResNet50 against ResNet50-Inception MultiLevel MultiScale**Question.** The complete per-line set, run on the multi-scale architecture as well as ResNet50, so the two can be compared on the same ground.**How everything else is held still.** Both models trained on the identical per-line train halves of the full 22-class per-line split and were scored on the identical test halves. Same preprocessing (raw 2448x2048 frame resized to 224x224 bilinear, no crop), same batch of 4, same 50 epochs, same exact-count stratified split with seed 42. A picture kept its side from the one split, so both models answered the same 2,527 test pictures. Every difference below is the architecture and nothing else.**Source of every number.** `run/line_results_multiscale_20260917/arch_vs_base.json`, written by the comparison script from the two runs' own confusion matrices. The cells below read that file, so re-running them reproduces these tables. The two runs are `runs/cnn/lines/` (ResNet50) and `runs/cnn/lines_multiscale/` (ResNet50-Inception MultiLevel MultiScale).

In [ ]:
import json, os, statistics as st_candidates = ["run/line_results_multiscale_20260917/arch_vs_base.json", os.path.join("..", "run/line_results_multiscale_20260917/arch_vs_base.json"), os.path.join(os.path.dirname(os.getcwd()), "run/line_results_multiscale_20260917/arch_vs_base.json")]_path = next(p for p in _candidates if os.path.exists(p))d = json.load(open(_path))lines, pooled = d["lines"], d["pooled"]print("read", _path)print("| Line | Test pictures | ResNet50 | MultiLevel MultiScale | Change | ResNet50 macro recall | MultiScale macro recall | Change |")print("|---|---|---|---|---|---|---|---|")for r in lines:    print("| {0} | {1:,} | {2:.2f}% | {3:.2f}% | {4:+.2f} | {5:.2f}% | {6:.2f}% | {7:+.2f} |".format(        r["line"], r["test_pictures"], r["accuracy_resnet50"] * 100, r["accuracy_multiscale"] * 100,        r["accuracy_gap"] * 100, r["macro_recall_resnet50"] * 100, r["macro_recall_multiscale"] * 100,        r["macro_recall_gap"] * 100))print("| All five | {0:,} | {1:.2f}% | {2:.2f}% | {3:+.2f} | {4:.2f}% | {5:.2f}% | {6:+.2f} |".format(    pooled["test_pictures"], pooled["accuracy_resnet50"] * 100, pooled["accuracy_multiscale"] * 100,    pooled["accuracy_gap"] * 100,    st.mean(r["macro_recall_resnet50"] for r in lines) * 100,    st.mean(r["macro_recall_multiscale"] for r in lines) * 100,    (st.mean(r["macro_recall_multiscale"] for r in lines)     - st.mean(r["macro_recall_resnet50"] for r in lines)) * 100))print()print("training minutes:", sum(r["train_minutes_resnet50"] for r in lines), "ResNet50 against",      sum(r["train_minutes_multiscale"] for r in lines), "multi-scale")

## What the table says- Pooled accuracy: **87.77% for ResNet50 against 89.47% for the multi-scale model, +1.70 points**.- Mean macro recall: **59.26% against 56.60%, -2.67 points**.- The two do not move together. Pooled accuracy counts pictures, so it follows the classes that hold the most pictures; macro recall counts classes equally, so it follows the thin ones.- Training time: 52 minutes for the five ResNet50 lines against 61 minutes for the five multi-scale lines, on the same box and the same data.The honest reading of the headline is that the multi-scale architecture buys about 1.7 points of accuracy while giving back about 2.7 points of macro recall: **better on the common defects, worse on the rare ones**. Which of the two matters is a product decision, not a measurement one, and the next section says which classes are involved.

## Where the accuracy change came from, class by classEach class-line cell contributes the change in the number of correct answers, divided by all 2,527 test pictures. The column adds up to the pooled accuracy change exactly, which turns the headline number into named classes instead of an unexplained average. One row per defect class, summed across the lines that carry it.

In [ ]:
import json, os, statistics as st_candidates = ["run/line_results_multiscale_20260917/arch_vs_base.json", os.path.join("..", "run/line_results_multiscale_20260917/arch_vs_base.json"), os.path.join(os.path.dirname(os.getcwd()), "run/line_results_multiscale_20260917/arch_vs_base.json")]_path = next(p for p in _candidates if os.path.exists(p))d = json.load(open(_path))lines, pooled = d["lines"], d["pooled"]print("read", _path)change = {}for pc in d["per_class"]:    change[pc["cls"]] = change.get(pc["cls"], 0) + pc["correct_change"]print("| Defect class | Correct answers gained or lost | Points of pooled accuracy |")print("|---|---|---|")for cls, share in sorted(pooled["attribution_by_class_share"].items(), key=lambda kv: -abs(kv[1])):    print("| {0} | {1:+d} | {2:+.2f} |".format(cls, change[cls], share * 100))print("| sum | | {0:+.4f} |".format(sum(pooled["attribution_by_class_share"].values()) * 100))print()print("pooled accuracy change: {0:+.4f} points".format(pooled["accuracy_gap"] * 100))print("class-line cells: better", pooled["cells_better"], "| worse", pooled["cells_worse"],      "| unchanged", pooled["cells_same"])

## How to read the class counts- Multi-scale recalled **more** in 20 class-line cells, **fewer** in 26, the same in 41. The accuracy gain comes from a minority of cells, and it does not come for free.- Among the cells with at least 20 test pictures, where a difference is worth reading at all: 26 cells, of which 10 improved and 9 got worse.- A cell with one or two test pictures can only score 0, 50 or 100 per cent, so a move there says nothing about an architecture. The table below carries the picture count in the third column so those cells can be skipped.**Largest recall losses** (line, class, change, test pictures): L26 Bubble Cluster -100% on 1; L26 Lens Off Center -100% on 2; L27 Bubble Scatter -100% on 1.**Largest recall gains**: L24 View Obstructed +100% on 6; L26 View Obstructed +80% on 10; L27 Bubble Cluster +67% on 3.

## Every class on every line, both architectures

In [ ]:
import json, os, statistics as st_candidates = ["run/line_results_multiscale_20260917/arch_vs_base.json", os.path.join("..", "run/line_results_multiscale_20260917/arch_vs_base.json"), os.path.join(os.path.dirname(os.getcwd()), "run/line_results_multiscale_20260917/arch_vs_base.json")]_path = next(p for p in _candidates if os.path.exists(p))d = json.load(open(_path))lines, pooled = d["lines"], d["pooled"]print("read", _path)print("| Line | Defect class | Test pictures | ResNet50 recall | MultiScale recall | Change |")print("|---|---|---|---|---|---|")for pc in sorted(d["per_class"], key=lambda x: (x["line"], -x["support"])):    print("| {0} | {1} | {2} | {3:.1f}% | {4:.1f}% | {5:+.1f}% |".format(        pc["line"], pc["cls"], pc["support"], pc["recall_resnet50"] * 100,        pc["recall_multiscale"] * 100, pc["recall_gap"] * 100))

## Caveats, stated with the result- One seed per architecture. The spread across seeds has not been measured on this data, so a difference of a point or two should not be treated as settled.- The training volumes are small: the thinnest line, L31, trains its model on 1,411 pictures, and several classes hold only a handful of test pictures.- Both models share one split, so the comparison between them is fair, but neither number should be read as an estimate of production accuracy.- Macro recall counts each class once, which is right for the rare classes and misleading for overall throughput. Both figures are reported together for that reason.